In [1]:
from pyspark.sql import functions as F, types as T, SparkSession

In [2]:
spark = SparkSession.builder.appName('ddl_test').getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/spark/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/home/glue_user/aws-glue-libs/jars/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [10]:
inventory_data = [
    {'ID': 'TR0013', 'OnHandQuantity': 278, 'OnHandQuantityDelta': 99, 'event_type': 'Outbound', 'event_datetime': '25/05/2020 00:25'},
    {'ID': 'TR0012', 'OnHandQuantity': 377, 'OnHandQuantityDelta': 31, 'event_type': 'Inbound', 'event_datetime': '24/05/2020 22:00'},
    {'ID': 'TR0011', 'OnHandQuantity': 346, 'OnHandQuantityDelta': 1, 'event_type': 'Outbound', 'event_datetime': '24/05/2020 15:01'},
    {'ID': 'TR0010', 'OnHandQuantity': 346, 'OnHandQuantityDelta': 102, 'event_type': 'Inbound', 'event_datetime': '25/04/2020 18:00'},
    {'ID': 'TR0009', 'OnHandQuantity': 246, 'OnHandQuantityDelta': 43, 'event_type': 'Inbound', 'event_datetime': '25/04/2020 02:00'},
    {'ID': 'TR0008', 'OnHandQuantity': 203, 'OnHandQuantityDelta': 2, 'event_type': 'Outbound', 'event_datetime': '25/02/2020 09:00'},
    {'ID': 'TR0007', 'OnHandQuantity': 205, 'OnHandQuantityDelta': 129, 'event_type': 'Outbound', 'event_datetime': '18/02/2020 08:00'},
    {'ID': 'TR0006', 'OnHandQuantity': 334, 'OnHandQuantityDelta': 1, 'event_type': 'Outbound', 'event_datetime': '18/02/2020 07:00'},
    {'ID': 'TR0005', 'OnHandQuantity': 335, 'OnHandQuantityDelta': 27, 'event_type': 'Outbound', 'event_datetime': '29/01/2020 05:00'},
    {'ID': 'TR0004', 'OnHandQuantity': 362, 'OnHandQuantityDelta': 120, 'event_type': 'Inbound', 'event_datetime': '31/12/2019 02:00'},
    {'ID': 'TR0003', 'OnHandQuantity': 242, 'OnHandQuantityDelta': 8, 'event_type': 'Outbound', 'event_datetime': '22/05/2019 00:50'},
    {'ID': 'TR0002', 'OnHandQuantity': 250, 'OnHandQuantityDelta': 250, 'event_type': 'Inbound', 'event_datetime': '20/05/2019 00:45'}
]

In [36]:
df_inventory = spark.createDataFrame(inventory_data).repartition(1)
df_inventory = df_inventory.withColumn("event_datetime", F.to_timestamp("event_datetime", "dd/MM/yyyy HH:mm"))
df_inventory.createOrReplaceTempView("inventory")
df_inventory.show()

+------+--------------+-------------------+-------------------+----------+
|    ID|OnHandQuantity|OnHandQuantityDelta|     event_datetime|event_type|
+------+--------------+-------------------+-------------------+----------+
|TR0013|           278|                 99|2020-05-25 00:25:00|  Outbound|
|TR0012|           377|                 31|2020-05-24 22:00:00|   Inbound|
|TR0011|           346|                  1|2020-05-24 15:01:00|  Outbound|
|TR0010|           346|                102|2020-04-25 18:00:00|   Inbound|
|TR0009|           246|                 43|2020-04-25 02:00:00|   Inbound|
|TR0008|           203|                  2|2020-02-25 09:00:00|  Outbound|
|TR0007|           205|                129|2020-02-18 08:00:00|  Outbound|
|TR0006|           334|                  1|2020-02-18 07:00:00|  Outbound|
|TR0005|           335|                 27|2020-01-29 05:00:00|  Outbound|
|TR0004|           362|                120|2019-12-31 02:00:00|   Inbound|
|TR0003|           242|  

In [35]:
spark.sql("""
          with inv as 
            (select *,
              datediff((select max(event_datetime) from inventory), event_datetime) as days_since_event
            from inventory
            order by event_datetime desc
          ),
          inv_age as (
            select 
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 0 and 90 and event_type='Inbound') inbound_0_90_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 0 and 90 and event_type='Outbound') outbound_0_90_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event > 90 and event_type='Inbound') - 
                  (select 
                  sum(OnHandQuantityDelta) from inv 
                  where days_since_event > 90 and event_type='Outbound') inhand_before_90,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 91 and 180 and event_type='Inbound') inbound_91_180_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 0 and 180 and event_type='Outbound') outbound_0_180_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event > 180 and event_type='Inbound') - 
                  (select 
                  sum(OnHandQuantityDelta) from inv 
                  where days_since_event > 180 and event_type='Outbound') inhand_before_180,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 181 and 270 and event_type='Inbound') inbound_180_270_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 0 and 270 and event_type='Outbound') outbound_0_270_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event > 270 and event_type='Inbound') - 
                  (select 
                  sum(OnHandQuantityDelta) from inv 
                  where days_since_event > 270 and event_type='Outbound') inhand_before_270,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 271 and 365 and event_type='Inbound') inbound_271_365_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event between 0 and 365 and event_type='Outbound') outbound_0_365_total,
              (select 
                sum(OnHandQuantityDelta) from inv 
                where days_since_event > 365 and event_type='Inbound') - 
                  (select 
                  sum(OnHandQuantityDelta) from inv 
                  where days_since_event > 365 and event_type='Outbound') inhand_before_365
            )
          
          select
            case 
              when inhand_before_90-outbound_0_90_total>0 then inbound_0_90_total
              else inbound_0_90_total+inhand_before_90-outbound_0_90_total
            end inbound_0_90_age,
            case 
              when inhand_before_180-outbound_0_180_total>0 then inbound_91_180_total
              else inbound_91_180_total+inhand_before_180-outbound_0_180_total
            end inbound_91_180_age,
            case 
              when inhand_before_270-outbound_0_270_total>0 then inbound_180_270_total
              else inbound_180_270_total+inhand_before_270-outbound_0_270_total
            end inbound_180_270_age,
            case 
              when inhand_before_365-outbound_0_365_total>0 then inbound_271_365_total
              else inbound_271_365_total+inhand_before_365-outbound_0_365_total
            end inbound_271_365_age
          from inv_age
""").show()

+----------------+------------------+-------------------+-------------------+
|inbound_0_90_age|inbound_91_180_age|inbound_180_270_age|inbound_271_365_age|
+----------------+------------------+-------------------+-------------------+
|             176|               103|               null|               null|
+----------------+------------------+-------------------+-------------------+



In [54]:
spark.stop()